In [2]:
import yfinance as yf
import pandas as pd

# Dependent variable: NVIDIA
stock = yf.download("NVDA", start="2024-09-01", end="2026-09-01", auto_adjust=True, progress=False)

# Independent variable 1: 10-Year Treasury yield
rates = yf.download("^TNX", start="2024-09-01", end="2026-09-01", auto_adjust=True, progress=False)

# Independent variable 2: Semiconductor sector index
sox = yf.download("^SOX", start="2024-09-01", end="2026-09-01", auto_adjust=True, progress=False)

# Combine into one dataframe (align by date)
df = pd.DataFrame({
    "NVDA_Close": stock["Close"].squeeze(),
    "Rate": rates["Close"].squeeze(),
    "SOX_Close": sox["Close"].squeeze()
}).dropna()

# Build returns (dependent variable) and independent variables
df["NVDA_return"] = df["NVDA_Close"].pct_change()
df["Rate_change"] = df["Rate"].diff()          
df["SOX_return"] = df["SOX_Close"].pct_change()

# Lag the independent variables so nothing looks forward
df["Rate_change_lag1"] = df["Rate_change"].shift(1)
df["SOX_return_lag1"] = df["SOX_return"].shift(1)

model_data = df[["NVDA_return", "Rate_change_lag1", "SOX_return_lag1"]].dropna().copy()
model_data.columns = ["NVIDIA_Return", "10YTRateChange", "SOXReturn"]
model_data.to_csv("financial_dataset.csv")

In this part of the code, I download throught the API of yahoo finance the historical of the returns of the NVIDIA stock, the dependent variable (Y), as well as the returns and changes in the 10 years treasury bond from the US and the semiconductor of Philadelphia Index, these two being the independent variables (X).
This code adjusts the price, to be able to reflect dividends and splits automatically. The code extracts the information and combines it into a single dataframe, organized according to the date. 
In addition, .dropna() allows to eliminate rows with missing data. This way we can work with the same data.
This specific code moves the values from the independent variables one row to the next day, in order to analyze if the changes of the Xs a day before, can explain the returns of NVIDIA the next day. 
At the end, the code drops the first row that now has missing values, then creates the data set CSV file.

In [3]:
print(model_data.shape)

(498, 3)


Here we can see the shape of the data set created, 498 rows and 3 columns, note how the date column is not read as an actual column.

In [10]:
model_data.head()

,NVIDIA_Return,10YTRateChange,SOXReturn
Date,,,
2024-09-05,0.009415,-0.076,0.002490
2024-09-06,-0.040854,-0.037,-0.005959
2024-09-09,0.035398,-0.021,-0.045167
2024-09-10,0.015309,-0.013,0.021545
2024-09-11,0.081499,-0.051,0.011866


We can check the first 5 rows of the data set, with the 3 main columns.

In [11]:
model_data.tail()

,NVIDIA_Return,10YTRateChange,SOXReturn
Date,,,
2026-08-25,0.021921,-0.034,-0.027018
2026-08-26,-0.015912,-0.065,0.014433
2026-08-27,0.087380,0.025,0.002002
2026-08-28,-0.045750,0.008,0.023333
2026-08-31,0.014847,0.048,-0.034717


We can also check the 5 last rows, we can observe we are anañyzing very recent data.

In [12]:
import statsmodels.api as sm

X = model_data[["10YTRateChange", "SOXReturn"]]
Y = model_data["NVIDIA_Return"]
X = sm.add_constant(X)

model = sm.OLS(Y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          NVIDIA_Return   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.6567
Date:                Mon, 07 Sep 2026   Prob (F-statistic):              0.519
Time:                        03:33:27   Log-Likelihood:                 1072.0
No. Observations:                 498   AIC:                            -2138.
Df Residuals:                     495   BIC:                            -2125.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0020      0.001      1.

As we can see, the R² = 0.003 and the adjusted R²  = -0.001. This means that the independent variables does not explain the variation of the NVIDIA results. The model does not tell us absolutely anything.
The F-Statistic of 0.519 reflect how the model is not significative at all.
The coefficients of the independent variables are not individually significative either, as their p-values are both bigger than 0.05. The coefficient of the intercept is not significant either. Overall, we cannot reject the null hypothesis, which means that the values of the variables from yesterday, do not have predictive power of todays NVIDIA return. The finding is consistent with the efficient market hypothesis (EMH), which suggests that the public information is already reflected in the market and should not predict the future. 

In [13]:
model_data2 = df[["NVDA_return", "Rate_change", "SOX_return"]].dropna().copy()
model_data2.columns = ["NVIDIA_Return", "RateChange", "SOXReturn_unlagged"]

X2 = model_data2[["RateChange", "SOXReturn_unlagged"]]
Y2 = model_data2["NVIDIA_Return"]
X2 = sm.add_constant(X2)

model2 = sm.OLS(Y2, X2).fit()
model_data2.to_csv("financial_dataset_unlagged.csv")
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:          NVIDIA_Return   R-squared:                       0.516
Model:                            OLS   Adj. R-squared:                  0.514
Method:                 Least Squares   F-statistic:                     264.7
Date:                Mon, 07 Sep 2026   Prob (F-statistic):           5.90e-79
Time:                        03:33:27   Log-Likelihood:                 1255.0
No. Observations:                 499   AIC:                            -2504.
Df Residuals:                     496   BIC:                            -2491.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0002      0

This other model is similar to the one before, the only difference is that the values of the variables are from the same day, in other words, how the data of the independent variables affect the NVIDIA stock returns from the same day. Note how here we have 499 observations, as we did not eliminate 1 extra.
As we can see, this model has an adjusted  R²  = 0.514, here the model has 51% power of explanation of the changes or returns of NVIDIA. In addition, the p-value of the F-statistic is extremely lower than 0.05, which reflects the model is highly significant. 
The coefficient of the intercept is not significant, as its p-value is higher than 0.05 (0.8). On the other side, the coeficients of Rate Change of the 10Y treasury bond (0.0498) is highly significant, lower than 0.05 (0.007); For every percentage point change in the 10YTB, the NVIDIA return changes 0.0498 points. On the other side, the coefficient of the index SOX Return (0.7469) is also highly significant (0.000), for every percentage point that the Index changes, NVIDIA returns 0.7469 percentage points.
Nevertheless, we need to highlight that rather than a predictive model, the explanation power of the model probably reflects how these variables move together, more like a co-movement relation, they react to the market context very similar.


Overall, after analyzing the results from excel, we found that the results are very similar, if not the same. For the results, all the results, coefficients, intercepts, R squares, p-value, etc. are practically the same, the only difference is tha in excel you get more information through the decimals.

With the purpose of reflection, the main difference that I identify is that in python, you can directly and automatically extract data and create data sets, while in excel you need to already have the excel file. 
For ease of use, for small data sets, and if you already have the data set, I think that using excel is easier, but for accesing information and larger data sets, python makes it more easy.
For level of cutomization, I feel that python opens you infinite possibilities, this is a simple excerside, but I think that you can really play with the program in orderto customize your results.  
For automation, python is better, as you can paste all your code in one single cell, or make changes and have results in seconds, while in excel there is a series of steps you have to follow, and doing that several times can be inconvenient.
For reproducibility, i feel that both are consistent with the results they give you. 
For depth in analysis, I feel that python offers you with infinite possibilities, while excel has finite options.
For ability to handle large datasets, definitely python is better, as excel has a row limit and could become slow while working with larger data sets.
For scalability, pyhton is better, as it would allow to leverage its use in bigger projects.
For application in the financial field, excel is good for academic purposes, and small practices, while python is a better tool to work with more advance models, handle larger data and extract real life and current information very fast.
Overall, I would say that the advantages of excel are familiarity, ease of use and that is an app that you probably already use in other things in your life, while the main disadvantages are limit of information and that you need to get the data set separately. For financial use, I would say that is better for simpler things, like budgeting.
For python the main advantages is that through an API allows you to get data very fast, can process and model big data, has a lot of possibilities for different models and customizations, between others, while for disadvantages, I would say that is hard to use if you are not familiar with it. For specific financial situations, I would say that is better for stock analysis, predictions, decision making, etc.

Overall, I prefer excel for simpler practices, like record expenses, make budgets or lists, even maybe create scenarios and forecasts, while I prefer python to build financial models regarding the stock market.